In [2]:
!pip install selenium

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.6 MB 5.6 MB/s eta 0:00:02
   -------------- ------------------------- 3.4/9.6 MB 8.7 MB/s eta 0:00:01
   -------------------- ------------------- 5.0/9.6 MB 8.9 MB/s eta 0:00:01
   ------------------------------ --------- 7.3/9.6 MB 9.2 MB/s eta 0:00:01
   -------------------------------------- - 9.2/9.6 MB 9.0 MB/s eta 0:00:01
   ---------------------------------------- 9.6/9.6 MB 8.9 MB/s  0:00:01

   ----- ----------------------------------  2/15 [sortedcontainers]
   ---------- -----------------------------  4/15 [wsproto]
   ---------------- -----------------------  6/15 [pysocks]
   --------------------- ------------------  8/15 [mypy-extensions]
   -------------------------- ------------- 10/15 [trio]
   -------------

In [4]:
!pip install webdriver_manager

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [webdriver_manager]
   ---------------------------------------- 2/2 [webdriver_manager]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [7]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def fetch_executive_order(order_number):
    url = f"https://www.federalregister.gov/executive-order/{order_number}"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    response = requests.get(url, headers=headers)
    print(f"Fetched URL: {url} (status code: {response.status_code})")
    if response.status_code != 200:
        print(f"Failed to fetch page: {url}")
        return
    soup = BeautifulSoup(response.text, 'html.parser')
    content_div = soup.find('div', id='fulltext_content_area')
    if not content_div:
        print("Content area not found. Saving HTML to 'debug_output.html' for inspection.")
        with open('debug_output.html', 'w', encoding='utf-8') as f:
            f.write(response.text)
        print(response.text[:500])
        return
    header = content_div.find('h1')
    title = header.get_text(strip=True) if header else "No header found."
    paragraphs = content_div.find_all('p')
    summary = "\n".join([p.get_text(strip=True) for p in paragraphs])
    return {
        'Executive Order #': order_number,
        'URL': url,
        'Title': title,
        'Summary': summary
    }

def fetch_range_of_orders(oldest, newest):
    results = []
    total = newest - oldest + 1
    start_time = time.time()
    for idx, order_number in enumerate(range(oldest, newest + 1), 1):
        print(f"Processing Executive Order {order_number} ({idx}/{total})...")
        iter_start = time.time()
        result = fetch_executive_order(str(order_number))
        iter_end = time.time()
        if result:
            results.append(result)
        # Estimate time remaining
        elapsed = iter_end - start_time
        avg_time = elapsed / idx
        remaining = total - idx
        est_remaining = avg_time * remaining
        print(f"Estimated time remaining: {est_remaining:.1f} seconds")
    if results:
        df = pd.DataFrame(results)
        df.to_excel('executive_order_output.xlsx', index=False)
        print("Output written to executive_order_output.xlsx")
    else:
        print("No valid executive orders found in the specified range.")

if __name__ == "__main__":
    oldest = int(input("Enter the oldest Executive Order number: "))
    newest = int(input("Enter the newest Executive Order number: "))
    fetch_range_of_orders(oldest, newest)

Enter the oldest Executive Order number:  14359
Enter the newest Executive Order number:  14371


Processing Executive Order 14359 (1/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14359 (status code: 200)
Estimated time remaining: 13.9 seconds
Processing Executive Order 14360 (2/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14360 (status code: 200)
Estimated time remaining: 12.8 seconds
Processing Executive Order 14361 (3/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14361 (status code: 200)
Estimated time remaining: 11.7 seconds
Processing Executive Order 14362 (4/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14362 (status code: 200)
Estimated time remaining: 10.4 seconds
Processing Executive Order 14363 (5/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14363 (status code: 200)
Estimated time remaining: 13.0 seconds
Processing Executive Order 14364 (6/13)...
Fetched URL: https://www.federalregister.gov/executive-order/14364 (status code: 200)
Estimated time remaining: 10.8 